# The Scenario: 
You are given three separate datasets: "Customer Profiles" (age, region, subscription tier), "Product Catalog" (category, weight, unit price), and "Order Transactions" (who bought what, when, and delivery times).

In [4]:
import pandas as pd

df1 = pd.read_csv('customers.csv')
df2 = pd.read_csv('products.csv')
df3 = pd.read_csv('orders_jan.csv')
df4 = pd.read_csv('orders_feb.csv')

print(df2.head())
print(df3.head())

  Product_ID Product_Category  Weight_kg  Unit_Price
0      P0001    Home & Garden       3.39      180.56
1      P0002           Sports       0.68      277.58
2      P0003      Electronics       0.64      204.59
3      P0004         Clothing       2.71      417.65
4      P0005           Sports      13.01      408.26
  Transaction_ID cust_ID Product_ID  Quantity  Order_Date  Delivery_Days  \
0         T00001   C0546      P0146         4  2024-01-26              2   
1         T00002   C0609      P0938         1  2024-01-22              6   
2         T00003   C0169      P0057         3  2024-01-07              6   
3         T00004   C0681      P0723         1  2024-01-10              1   
4         T00005   C0663      P0780         3  2024-01-28              1   

  Warehouse_Route_ID  
0              WH-A1  
1              WH-C4  
2              WH-B3  
3              WH-A9  
4              WH-A7  


### Combining Datasets:
You must use pd.concat() to stitch together the order transactions from "January" and "February". 

In [5]:
df_combined = pd.concat([df3, df4], ignore_index=True)

print(df_combined.head())

  Transaction_ID cust_ID Product_ID  Quantity  Order_Date  Delivery_Days  \
0         T00001   C0546      P0146         4  2024-01-26              2   
1         T00002   C0609      P0938         1  2024-01-22              6   
2         T00003   C0169      P0057         3  2024-01-07              6   
3         T00004   C0681      P0723         1  2024-01-10              1   
4         T00005   C0663      P0780         3  2024-01-28              1   

  Warehouse_Route_ID  
0              WH-A1  
1              WH-C4  
2              WH-B3  
3              WH-A9  
4              WH-A7  


Then, use pd.merge() to join the combined 'Order Transactions' table with the 'Product Catalog' table so you know the category and price of the item purchased, acting like a SQL left join.

In [6]:
df_merge = pd.merge(df_combined, df2, on='Product_ID', how='left')

print(df_merge.head(5))

  Transaction_ID cust_ID Product_ID  Quantity  Order_Date  Delivery_Days  \
0         T00001   C0546      P0146         4  2024-01-26              2   
1         T00002   C0609      P0938         1  2024-01-22              6   
2         T00003   C0169      P0057         3  2024-01-07              6   
3         T00004   C0681      P0723         1  2024-01-10              1   
4         T00005   C0663      P0780         3  2024-01-28              1   

  Warehouse_Route_ID Product_Category  Weight_kg  Unit_Price  
0              WH-A1    Home & Garden      11.97      403.69  
1              WH-C4             Toys      11.11      152.34  
2              WH-B3         Clothing      10.97        8.16  
3              WH-A9      Electronics      11.15      488.40  
4              WH-A7           Sports       7.34      476.43  


### Modifying DataFrames:
You create a new calculated column called Total_Revenue by multiplying the Quantity column by the Unit_Price column. 

In [7]:
df_merge['Total_Revenue'] = df_merge['Quantity'] * df_merge['Unit_Price']

print(df_merge.head(5))

  Transaction_ID cust_ID Product_ID  Quantity  Order_Date  Delivery_Days  \
0         T00001   C0546      P0146         4  2024-01-26              2   
1         T00002   C0609      P0938         1  2024-01-22              6   
2         T00003   C0169      P0057         3  2024-01-07              6   
3         T00004   C0681      P0723         1  2024-01-10              1   
4         T00005   C0663      P0780         3  2024-01-28              1   

  Warehouse_Route_ID Product_Category  Weight_kg  Unit_Price  Total_Revenue  
0              WH-A1    Home & Garden      11.97      403.69        1614.76  
1              WH-C4             Toys      11.11      152.34         152.34  
2              WH-B3         Clothing      10.97        8.16          24.48  
3              WH-A9      Electronics      11.15      488.40         488.40  
4              WH-A7           Sports       7.34      476.43        1429.29  


Use .drop() to remove redundant warehouse routing IDs, and use .rename() to fix inconsistently capitalized columns (e.g., changing cust_ID to Customer_ID).

In [8]:
df_final = df_merge.drop(columns=['Warehouse_Route_ID']).rename(columns={
    'cust_ID': 'Customer_ID',
    'Product_ID': 'Product_ID',
})

print(df_final.head(5))

  Transaction_ID Customer_ID Product_ID  Quantity  Order_Date  Delivery_Days  \
0         T00001       C0546      P0146         4  2024-01-26              2   
1         T00002       C0609      P0938         1  2024-01-22              6   
2         T00003       C0169      P0057         3  2024-01-07              6   
3         T00004       C0681      P0723         1  2024-01-10              1   
4         T00005       C0663      P0780         3  2024-01-28              1   

  Product_Category  Weight_kg  Unit_Price  Total_Revenue  
0    Home & Garden      11.97      403.69        1614.76  
1             Toys      11.11      152.34         152.34  
2         Clothing      10.97        8.16          24.48  
3      Electronics      11.15      488.40         488.40  
4           Sports       7.34      476.43        1429.29  


### Applying Functions: 
You write a custom function (or use a lambda) and apply it via .apply() to categorize the Delivery_Days column into delivery performance buckets: "Early" (1-2 days), "On-Time" (3-4 days), or "Delayed" (5+ days).

In [9]:
def delivery(days):
    if days <= 2:
        return "Early"
    elif days <= 4:
        return "On-Time"
    else:
        return "Delayed"

df_final['Delivery_Performance'] = df_final['Delivery_Days'].apply(delivery)

print(df_final[['Transaction_ID', 'Delivery_Days', 'Delivery_Performance']].head())

  Transaction_ID  Delivery_Days Delivery_Performance
0         T00001              2                Early
1         T00002              6              Delayed
2         T00003              6              Delayed
3         T00004              1                Early
4         T00005              1                Early


### Grouping and Aggregation:
Using .groupby(), you group the data by Product_Category and use .agg() to find the total revenue generated, the average delivery days, and the unique count of Customer_IDs who purchased from that category.

In [10]:
category_summary = df_final.groupby('Product_Category').agg(
    Total_Revenue=('Total_Revenue', 'sum'),
    Average_Delivery_Days=('Delivery_Days', 'mean'),
    Unique_Customers=('Customer_ID', 'nunique')
)

print(category_summary)

                  Total_Revenue  Average_Delivery_Days  Unique_Customers
Product_Category                                                        
Clothing              236216.30               4.005764               289
Electronics           291529.29               3.960688               333
Home & Garden         318433.12               4.009324               342
Sports                311298.28               3.847255               346
Toys                  300972.12               3.871859               316


### Reshaping and Pivoting: 
Finally, you use pd.pivot_table() to create a business-reporting matrix showing "Customer Region" as rows, "Delivery Performance" (Early/On-Time/Delayed) as columns, and the "Total Revenue" as the values.

In [11]:
df_final = pd.merge(df_final, df1, on='Customer_ID', how='left')

revenue_pivot = pd.pivot_table(
    df_final, 
    values='Total_Revenue', 
    index='Region', 
    columns='Delivery_Performance', 
    aggfunc='sum',
    fill_value=0
)

print(revenue_pivot)

Delivery_Performance    Delayed      Early    On-Time
Region                                               
East                  165532.82   91495.83  102829.30
North                 168913.17  115843.80  125378.86
South                 157531.94  112139.58  102640.26
West                  129061.42  111503.35   75578.78


In [12]:
df_final

,Transaction_ID,Customer_ID,Product_ID,Quantity,Order_Date,Delivery_Days,Product_Category,Weight_kg,Unit_Price,Total_Revenue,Delivery_Performance,Age,Region,Subscription_Tier
0,T00001,C0546,P0146,4,2024-01-26,2,Home & Garden,11.97,403.69,1614.76,Early,70,South,Premium
1,T00002,C0609,P0938,1,2024-01-22,6,Toys,11.11,152.34,152.34,Delayed,41,North,Basic
2,T00003,C0169,P0057,3,2024-01-07,6,Clothing,10.97,8.16,24.48,Delayed,49,East,Premium
3,T00004,C0681,P0723,1,2024-01-10,1,Electronics,11.15,488.40,488.40,Early,42,South,Basic
4,T00005,C0663,P0780,3,2024-01-28,1,Sports,7.34,476.43,1429.29,Early,52,South,Basic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,T01996,C0637,P0623,3,2024-02-19,3,Toys,7.38,298.92,896.76,On-Time,43,East,Plus
1996,T01997,C0716,P0210,3,2024-02-09,4,Toys,4.08,55.91,167.73,On-Time,70,North,Basic
1997,T01998,C0573,P0513,1,2024-02-09,1,Toys,2.94,85.87,85.87,Early,33,North,Basic
1998,T01999,C0762,P0058,3,2024-02-13,2,Home & Garden,14.51,489.29,1467.87,Early,24,North,Premium
